In [2]:
# 1. 필요한 라이브러리를 불러온다
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    Trainer, 
    TrainingArguments
)

In [ ]:
# 2. 준비된 학습용 데이터를 불러온다.
df = pd.read_csv("./data/review_data.csv", encoding='cp949')

In [4]:
df.head(3)

,text,labels
0,배우들 연기도 너무 좋았어요.,1
1,스토리가 탄탄하고 연출도 훌륭했어요.,1
2,정말 감동적인 영화였습니다. 눈물이 멈추질 않았어요.,1


In [ ]:
# 3. 트레닝/테스트 데이터로 분할한다.
train_df, test_df = train_test_split(
        df, 
        test_size=0.2, 
        stratify=df['labels'], 
        random_state=0
)

In [ ]:
# 4. 허깅페이스의 Dataset 형식으로 변환한다.
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [7]:
print(train_dataset)

Dataset({
    features: ['text', 'labels', '__index_level_0__'],
    num_rows: 80
})


In [8]:
print(vars(train_dataset))

{'_info': DatasetInfo(features={'text': Value('large_string'), 'labels': Value('int64'), '__index_level_0__': Value('int64')}), '_split': None, '_indexes': {}, '_data': InMemoryTable
text: large_string
labels: int64
__index_level_0__: int64
----
text: [["배우의 감정 연기가 정말 인상 깊었어요.","개연성이 너무 떨어져요.","시간 아깝고 돈 아깝네요.","분위기가 정말 좋았어요.","전체적으로 산만한 느낌이에요.",...,"연기력이 영화의 완성도를 높였어요.","배우 연기가 너무 어색해서 몰입이 안 됐어요.","배우 활용을 잘 못한 느낌이에요.","분위기와 음악이 잘 어우러졌어요.","완성도가 많이 떨어져 보여요."]]
labels: [[1,0,0,1,0,...,1,0,0,1,0]]
__index_level_0__: [[10,63,57,32,70,...,33,51,87,46,78]], '_indices': None, '_format_type': None, '_format_kwargs': {}, '_format_columns': None, '_output_all_columns': False, '_fingerprint': '3ff51378b8273a81'}


In [9]:
train_dataset[0]

{'text': '배우의 감정 연기가 정말 인상 깊었어요.', 'labels': 1, '__index_level_0__': 10}

In [ ]:
# 5. 학습에 사용할 LLM 모델과 토크나이저를 불러온다.
model_id = "beomi/kcbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, 
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3364.98it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [ ]:
# 6. Dataset.map()의 내부 포함되어 입력될 문자열 시퀀스 데이터를 토큰 ID로 바꿔주는 전처리 함수이다.
def preprocess(data):
    return tokenizer(
        data["text"], 
        padding = "max_length",
        truncation = True, 
        max_length = 64)

In [12]:
train_dataset = train_dataset.map(
    preprocess, 
    batched = True,
    remove_columns = ["text", "__index_level_0__"]
)

test_dataset = test_dataset.map(
    preprocess, 
    batched=True,
    remove_columns=["text", "__index_level_0__"]
)

Map: 100%|██████████| 20/20 [00:00<00:00, 314.25 examples/s]


In [13]:
print(train_dataset)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 80
})


In [14]:
print(train_dataset[0])

{'labels': 1, 'input_ids': [2, 10631, 4042, 9969, 11219, 4009, 8050, 11662, 429, 16849, 17, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [ ]:
# 7. 학습 설정에 대한 모든 규칙을 담는다.
training_args = TrainingArguments(
    output_dir="./saved_models/basic_sentiment",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,

    logging_strategy="epoch",

    use_cpu=True
)

In [ ]:
# 8. 학습기를 만들고 훈련한다.
def compute_metrics(predict):
    preds = np.argmax(predict.predictions, axis=1)
    acc = np.mean(preds == predict.label_ids)
    return {"accuracy": acc}

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.664177,0.634925,0.650000
2,0.263874,0.951073,0.700000
3,0.075014,0.289855,0.900000
4,0.013663,0.424590,0.900000
5,0.002191,0.576898,0.900000


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.62s/it]


TrainOutput(global_step=50, training_loss=0.20378376737236978, metrics={'train_runtime': 486.3904, 'train_samples_per_second': 0.822, 'train_steps_per_second': 0.103, 'total_flos': 13155552768000.0, 'train_loss': 0.20378376737236978, 'epoch': 5.0})

In [ ]:
# 9. 학습을 마친 모델과 새로운 데이터를 활용해 예측 성능을 확인한다.
# 테스트 예측
test_texts = [
    # 긍정 데이터
    "전체적인 분위기가 좋아서 편하게 볼 수 있었어요.",
    "스토리는 평범했지만 연출 덕분에 재미있었어요.",
    "배우들의 연기가 자연스러워서 몰입이 잘 됐어요.",
    "큰 기대 없이 봤는데 생각보다 괜찮았어요.",
    "잔잔하지만 끝나고 나서 여운이 남는 영화였어요.",

    # 부정 데이터
    "이야기가 늘어져서 중간부터 집중이 안 됐어요.",
    "연출이 과해서 오히려 몰입을 방해했어요.",
    "캐릭터 행동이 이해되지 않아서 답답했어요.",
    "분위기는 잡으려는 것 같은데 내용이 부족했어요.",
    "전체적으로 뭔가 아쉬운 느낌이 많이 남았어요."
]

inputs = tokenizer(
    test_texts, 
    return_tensors="pt", 
    padding=True, 
    truncation=True, 
    max_length=64
)

In [ ]:
# 10. 베스트 모델을 활용해 예측한다.
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
preds = torch.argmax(outputs.logits, dim=1)
print("예측 결과:", preds.tolist())

labels_target = torch.tensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
accuracy = (preds == labels_target).float().mean()
print("Accuracy:", accuracy.item())

예측 결과: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
Accuracy: 1.0
